In [6]:
import pandas as pd
import numpy as np

In [7]:
df = pd.read_csv("vaccine_shipments_raw.xls")

In [8]:
df.head()

,shipment_id,container_id,vaccine_type,batch_id,quantity,source,destination,departure_time,expected_arrival,min_temperature,max_temperature
0,SHP0001,CNT1001,Pfizer,BTH05001,10000.0,Kochi,Mumbai,2026-08-08 13:15:00,2026-08-09 14:15:00,2,8
1,SHP0002,CNT1002,Sputnik V,BTH05002,2500.0,Bengaluru,Delhi,2026-08-02 06:00:00,2026-08-03 21:00:00,2,8
2,SHP0003,CNT1003,Sputnik V,BTH05003,5000.0,Thiruvananthapuram,Kochi,2026-08-18 12:45:00,2026-08-20 04:45:00,2,8
3,SHP0004,CNT1004,Sputnik V,BTH05004,3000.0,Mumbai,Kochi,2026-08-07 04:45:00,2026-08-09 11:45:00,2,8
4,SHP0005,CNT1005,Moderna,BTH05005,3500.0,Hyderabad,Pune,2026-08-04 08:45:00,2026-08-05 08:45:00,2,8


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 501 entries, 0 to 500
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   shipment_id       501 non-null    object 
 1   container_id      501 non-null    object 
 2   vaccine_type      498 non-null    object 
 3   batch_id          501 non-null    object 
 4   quantity          499 non-null    float64
 5   source            501 non-null    object 
 6   destination       501 non-null    object 
 7   departure_time    499 non-null    object 
 8   expected_arrival  501 non-null    object 
 9   min_temperature   501 non-null    int64  
 10  max_temperature   501 non-null    int64  
dtypes: float64(1), int64(2), object(8)
memory usage: 43.2+ KB


In [11]:
import pandas as pd
import numpy as np

# Load raw file
df = pd.read_csv('vaccine_shipments_raw.xls')

# 1. Deduplicate
df = df.drop_duplicates()

# 2. String Cleaning (Whitespace & Title Casing)
string_cols = ['vaccine_type', 'source', 'destination']
for col in string_cols:
    df[col] = df[col].astype(str).str.strip().str.title()
    df[col] = df[col].replace('Nan', np.nan)

# 3. Handle Quantity Errors & Outliers
df['quantity'] = df['quantity'].abs()  # Fix negative values (-500 -> 500)
df['quantity'] = df.groupby('vaccine_type')['quantity'].transform(lambda x: x.fillna(x.median()))

# 4. Correct Temperature Anomalies
df.loc[df['max_temperature'] > 8, 'max_temperature'] = 8

# 5. Standardize Datetime Fields
for date_col in ['departure_time', 'expected_arrival']:
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce', format='mixed')

# Save cleaned output
df.to_csv('vaccine_shipments_cleaned.csv', index=False)
print("Data cleaning completed successfully.")

Data cleaning completed successfully.


In [12]:
import pandas as pd
import numpy as np

# 1. Load raw file
df = pd.read_csv('vaccine_shipments_raw.xls')

# 2. Deduplicate
df = df.drop_duplicates()

# 3. String Cleaning (Whitespace & Title Casing)
string_cols = ['vaccine_type', 'source', 'destination']
for col in string_cols:
    df[col] = df[col].astype(str).str.strip().str.title()
    df[col] = df[col].replace('Nan', np.nan)

# Fill missing vaccine types FIRST so groupby works properly
df['vaccine_type'] = df['vaccine_type'].fillna('Unknown')

# 4. Handle Quantity Errors & Outliers
df['quantity'] = df['quantity'].abs()  # Convert negative values (-500 -> 500)
df['quantity'] = df.groupby('vaccine_type')['quantity'].transform(lambda x: x.fillna(x.median()))

# 5. Correct Temperature Anomalies
df.loc[df['max_temperature'] > 8, 'max_temperature'] = 8

# 6. Standardize Datetime Fields
for date_col in ['departure_time', 'expected_arrival']:
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce', format='mixed')

# Option A: Drop unresolvable missing date rows
df_clean = df.dropna(subset=['departure_time', 'expected_arrival']).copy()

# Save complete cleaned output
df_clean.to_csv('vaccine_shipments_cleaned.csv', index=False)
print("Updated pipeline completed. Total rows:", len(df_clean))
print("Remaining Nulls:\n", df_clean.isnull().sum())

Updated pipeline completed. Total rows: 497
Remaining Nulls:
 shipment_id         0
container_id        0
vaccine_type        0
batch_id            0
quantity            0
source              0
destination         0
departure_time      0
expected_arrival    0
min_temperature     0
max_temperature     0
dtype: int64
